# brand_probe — votre marque est-elle dans les poids des modèles ?

Kit à emporter de la conférence TeknSEO *« GEO : le vrai du faux, mesuré par les log-probs »*.

Trois cellules : installer, (optionnel) se connecter à Hugging Face, mesurer.

**Sans GPU** (`Runtime → Change runtime type → CPU` ou `T4`) : les trois modèles de scène, 1 à 2,6 Md, environ 2 minutes de chargement puis 20 s par marque.  
**Avec un GPU 24 Go** (Colab Pro : `L4` ou `A100`) : ajoutez `qwen-7b`, `llama-8b`, `gemma-9b` dans `MODELS`.

> ⚠️ Ce kit ne mesure **pas** la citation par ChatGPT ou Gemini. La corrélation mesurée entre présence dans les poids et fréquence de citation est de +0,1 : être dans les poids est une condition d'entrée, pas une cause. Voir le README.

In [ ]:
#@title 1 · Installation (une minute)
#@markdown Adresse du script. Laissez la valeur par défaut, ou pointez sur votre copie.
SCRIPT_URL = "https://raw.githubusercontent.com/rafaleur/TeknSEO2026-PSauve/main/brand_probe.py"  #@param {type:"string"}

import subprocess, sys, urllib.request

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).returncode == 0
print("GPU détecté" if gpu else "Pas de GPU : modèles 1–2,6 Md seulement")
# Colab fournit déjà torch (CUDA). On n'installe que le reste.
!pip -q install "transformers>=4.57" "accelerate>=1.0" "huggingface_hub>=0.35" "safetensors>=0.6"
urllib.request.urlretrieve(SCRIPT_URL, "brand_probe.py")
print("brand_probe.py téléchargé")

In [ ]:
#@title 2 · (Optionnel) Hugging Face — nécessaire pour Llama et Gemma
#@markdown Qwen se passe de compte. Pour **Llama** (Meta) et **Gemma** (Google) :
#@markdown 1. compte sur https://huggingface.co ;
#@markdown 2. accepter la licence sur la page du modèle (https://huggingface.co/meta-llama/Llama-3.2-1B, https://huggingface.co/google/gemma-2-2b, et pour le GPU https://huggingface.co/meta-llama/Llama-3.1-8B, https://huggingface.co/google/gemma-2-9b) ;
#@markdown 3. créer un jeton *Read* sur https://huggingface.co/settings/tokens et le coller ci-dessous — ou le ranger dans les *Secrets* de Colab (icône clé, nom `HF_TOKEN`), la cellule le trouvera.
HF_TOKEN = ""  #@param {type:"string"}

import os
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = ""
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import whoami
    print("Connecté à Hugging Face en tant que", whoami()["name"])
else:
    print("Pas de jeton : seuls les modèles Qwen seront chargeables.")

In [ ]:
#@title 3 · Mesurer
BRANDS = "Ahrefs, Oncrawl"  #@param {type:"string"}
#@markdown Clé de catégorie (`seo`, `analytics`, `email`, `support`, `survey`, `social`, `crm`, `assurance`, `cosmetique`, `ameublement`, `bricolage`) **ou** une phrase libre : `is a software tool for rank tracking` / `est une banque en ligne`.
CATEGORY = "seo"  #@param {type:"string"}
LANG = "en"  #@param ["en", "fr"]
#@markdown Modèles, séparés par des virgules. Sans GPU : `qwen-1.5b`, `llama-1b`, `gemma-2b`. Avec 24 Go : `qwen-7b`, `llama-8b`, `gemma-9b`.
MODELS = "qwen-1.5b"  #@param {type:"string"}
CORPUS = True  #@param {type:"boolean"}
#@markdown Requête infini-gram de remplacement si le nom est ambigu (ex. `Brevo AND emailing`). Vide = le nom.
CORPUS_QUERY = ""  #@param {type:"string"}

import json, sys
sys.argv = ["brand_probe.py"]
import importlib, brand_probe
importlib.reload(brand_probe)

brands = [b.strip() for b in BRANDS.split(",") if b.strip()]
models = [m.strip() for m in MODELS.split(",") if m.strip()]
reports = brand_probe.probe(
    brands, CATEGORY, LANG, models=models, corpus=CORPUS,
    corpus_query=CORPUS_QUERY or None, log=lambda m: print(m, file=sys.stderr),
)
for r in reports:
    print(brand_probe.render(r))

from dataclasses import asdict
with open("brand_probe.json", "w", encoding="utf-8") as f:
    json.dump([asdict(r) for r in reports], f, ensure_ascii=False, indent=2)
print("Rapport complet : brand_probe.json (panneau Fichiers, à gauche)")

## Lire le rapport

- **Marge** : log-probabilité moyenne de « *Marque* is a software tool for search engine optimization » moins celle de la meilleure des quatre fausses phrases (café italien, poisson d'eau douce, village du sud de la France, danse folklorique). Positif = le modèle préfère la vraie catégorie.
- **Plancher** : la marge la plus haute obtenue par quatre noms inventés, *dans ce run, sur ce modèle*. Une marque sous le plancher est indistinguable d'un nom qui n'existe pas.
- **Lift** d'un terme : combien le mot est plus probable après votre marque qu'après un nom inventé, en nats (+7 ≈ mille fois). Les termes à lift ≈ 0 (« new », « company ») sont ce que le modèle dit de tout le monde.
- **Corpus** : occurrences par milliard de mots dans C4 (2019), DCLM (2023), Dolma (2024). Le script signale quand le nom est en réalité un autre mot (« Brevo » ⊂ « Brevoort ») et quand les corpus sont trop anciens pour une marque récente.

Détails, pièges et option GPU : le `README.md` du kit.